# CNN with PyTorch for FashionMNIST

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

In [ ]:
# Check the ability of GPU device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
torch.manual_seed(42)

In [ ]:
transform = transforms.ToTensor()

train_data = datasets.FashionMNIST(
    root="../../datasets/",
    train = True,
    download = True,
    transform=transform,
)

test_data = datasets.FashionMNIST(
    root="../../datasets/",
    train=False,
    download=True,
    transform=transform
)


train_loader = DataLoader(train_data, batch_size=64*4, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_data, batch_size=len(test_data), shuffle=False, pin_memory=True)

In [ ]:
feature, target = train_data[0]
feature.shape

In [ ]:
# Model Definition
class CNN(nn.Module):
    def __init__(self, input_channels):
        super().__init__()
        
        self.features = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=3, padding='same'),
                # input channel 1, filter count 32, filter kernel 3x3, padding no loss
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
                # 2x2 kernel dim, 2 stride
                
            nn.Conv2d(32, 64, kernel_size=3, padding='same'),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        ) # CNN Container
        self.classifier = nn.Sequential(
            nn.Flatten(),
            
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.35),
            
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.35),
            
            nn.Linear(64, 10),
            # Softmax activation applied automatically by optimizer
        ) # ANN Container
        
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        
        return x

In [ ]:
learning_rate = 0.01
epochs = 20

In [ ]:
model = CNN(1)
model.to(device=device)

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

In [ ]:
# Training loop
for epoch in range(epochs):
    epoch_loss = 0
    for batch_features, batch_labels in train_loader                        :
        
        # Move batch_features and batch_labels to GPU
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        
        # Forward pass
        outputs = model(batch_features)
        
        # Calculate Loss
        loss = criterion(outputs, batch_labels)
        epoch_loss += loss.item()
        # Clear gradients
        optimizer.zero_grad()
        
        # Backward Pass
        loss.backward()
        
        # Update Gradients
        optimizer.step()
        
    # len(train_loader) returns batch_count
    print(f"Epoch :{epoch + 1}, Loss: {epoch_loss/len(train_loader)}")

In [ ]:
# Evaluation
test_features, test_labels = next(iter(test_loader))
test_features, test_labels = test_features.to(device), test_labels.to(device)


# Prepare model for evaluation mode
model.eval()

with torch.no_grad():
    logits = model(test_features)
    
    # Convert to numpy for using scikit learn's methods
    preds = logits.argmax(dim=1).cpu().numpy()
    labels = test_labels.cpu().numpy()
    
    accuracy = accuracy_score(labels, preds)
    precision = precision_score(labels, preds, average='macro')
    recall = recall_score(labels, preds, average='macro')
    f1 = f1_score(labels, preds, average='macro')
    
print(f"Accuracy: {accuracy}, Precision: {precision}, Recall: {recall}, F1 Score: {f1}")